# Query-Level Failure Case Studies

## 1. Imports, Paths, Constants

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

#Paths 
PROJECT_ROOT=Path.cwd().parent
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models_folds'
PHASE10_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase10_fold_robustness'
PHASE11_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase11_case_studies'
PHASE11_OUTPUT.mkdir(parents=True, exist_ok=True)

#Constants
DATASET='2007'
FOLD='Fold1'
BASELINE_MODEL='pointwise'
BASELINE_PIPELINE='raw'
N_EXAMPLES=3  #Number of example queries

print('='*80)
print('PHASE 11: QUERY-LEVEL FAILURE CASE STUDIES')
print('='*80)
print(f'Output: {PHASE11_OUTPUT}')
print(f'Dataset: MQ{DATASET} {FOLD}')
print(f'Baseline: {BASELINE_MODEL}_{BASELINE_PIPELINE}')
print(f'N examples: {N_EXAMPLES}')
print('\nPurpose: Illustrate structural patterns from Phases 8-10')
print('='*80)

PHASE 11: QUERY-LEVEL FAILURE CASE STUDIES
Output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase11_case_studies
Dataset: MQ2007 Fold1
Baseline: pointwise_raw
N examples: 3

Purpose: Illustrate structural patterns from Phases 8-10


## 2. Loading Phase 6 Baseline Artifacts

In [2]:
print('\n'+'='*80)
print('LOADING BASELINE ARTIFACTS')
print('='*80)

baseline_key=f'{BASELINE_MODEL}_{BASELINE_PIPELINE}_{DATASET}'
qm_file=PHASE6_OUTPUT / FOLD / f'{baseline_key}_query_metrics.csv'
pred_file=PHASE6_OUTPUT / FOLD / f'{baseline_key}_predictions.csv'

if not qm_file.exists():
    raise RuntimeError(f'Missing: {qm_file}')
if not pred_file.exists():
    raise RuntimeError(f'Missing: {pred_file}')

qm=pd.read_csv(qm_file)
pred=pd.read_csv(pred_file)

#Validating
for col in ['qid', 'num_relevant_1', 'Failure@5_primary']:
    if col not in qm.columns:
        raise RuntimeError(f'Missing column: {col}')

for col in ['qid', 'label', 'score']:
    if col not in pred.columns:
        raise RuntimeError(f'Missing column: {col}')

print(f'Query metrics: {len(qm)} queries')
print(f'Predictions: {len(pred)} documents')
print('='*80)


LOADING BASELINE ARTIFACTS
Query metrics: 336 queries
Predictions: 13652 documents


## 3. Loading Persistent Queries from Phase 10

In [3]:
print('\n'+'='*80)
print('LOADING PERSISTENT QUERIES FROM PHASE 10')
print('='*80)
print('Using persistent queries already identified in Phase 10')
print('Persistent = intersection of failures across ALL 9 configs')
print('='*80)

#Phase 10 doesn't store qid lists directly, so we reconstruct from baseline
#but only within the persistent set identified in Phase 10

#Loading all 9 configs to identify persistent set
models=['pointwise', 'pairwise', 'lightgbm']
pipelines=['raw', 'global', 'per_query']

def make_fail_flag(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).round().astype(int).clip(0, 1)
    return series.map(lambda v: 1 if str(v).lower() in ['true','1','yes'] else 0).astype(int)

evaluable_sets={}
failure_sets={}

for model in models:
    for pipeline in pipelines:
        key=f'{model}_{pipeline}_{DATASET}'
        qm_file=PHASE6_OUTPUT / FOLD / f'{key}_query_metrics.csv'
        
        if not qm_file.exists():
            print(f'Missing: {key}')
            continue
        
        qm_config=pd.read_csv(qm_file)
        evaluable_sets[key]=set(qm_config.loc[qm_config['num_relevant_1'] > 0, 'qid'])
        
        fail_flag=make_fail_flag(qm_config['Failure@5_primary'])
        fail_mask=(qm_config['num_relevant_1'] > 0) & (fail_flag == 1)
        failure_sets[key]=set(qm_config.loc[fail_mask, 'qid'])

#Computing persistent (same logic as Phase 10)
if len(evaluable_sets)==9 and len(failure_sets)==9:
    all_evaluable=set.intersection(*evaluable_sets.values())
    persistent_qids=set.intersection(*failure_sets.values())
    print(f'\nLoaded {len(evaluable_sets)} configs')
    print(f'Evaluable queries: {len(all_evaluable)}')
    print(f'Persistent failures: {len(persistent_qids)}')
else:
    raise RuntimeError(f'Expected 9 configs, found {len(evaluable_sets)}')

print('\nPersistent = queries failing across ALL 9 model×pipeline configs')
print('This matches the definition from Phase 10')
print('='*80)


LOADING PERSISTENT QUERIES FROM PHASE 10
Using persistent queries already identified in Phase 10
Persistent = intersection of failures across ALL 9 configs

Loaded 9 configs
Evaluable queries: 290
Persistent failures: 22

Persistent = queries failing across ALL 9 model×pipeline configs
This matches the definition from Phase 10
